# Example for 1 billion molecules clustered from the Enamine Database

In [1]:
from rdkit import rdBase
blocker = rdBase.BlockLogs() # Supress RDKIT Warnings
from chelombus.utils import format_time


# Getting Training data
Enamine Database can be downloaded from `https://enamine.net/compound-collections/real-compounds/real-database`

In [ ]:
from chelombus import DataStreamer
from chelombus import FingerprintCalculator
from tqdm import tqdm
from chelombus.utils import save_chunk

# These can be useful to make the transformation from .txt/sdf format to MQN fingerprint easier
stream = DataStreamer()  
fp_calc = FingerprintCalculator()

chunksize = 10_000_000
path ='~/Downloads/EnamineFiles/1B_Enamine_Molecules.txt'  # Replace with the actual path to the file
smiles_generator = stream.parse_input(
     path,
     chunksize=chunksize, # chunk size, whatever fits comfortably in RAM. This process is paralellized 
     smiles_col=1 # column where the SMILES strings are located
     ) 

# For 1 billion molecules, around 100M should be enough to train the encoder and cluster algorithm
iterations = int(100_000_000 / chunksize)
for i, chunk in tqdm(enumerate(smiles_generator), desc='Calculating Fingerprints', total=iterations):
    chunk_fp = fp_calc.FingerprintFromSmiles(chunk, fp="mqn") # Calculate MQN fingerprint
    output_path = 'tmp/' # Define your output path, make sure enough space is available
    save_chunk(chunk_fp,
               name='enamine-chunk',
               chunk_index=i,  # Each file will be named name+chunk_index.npy
               output_dir=output_path)

    if i == iterations:
       break


 At this point we've only transformed our SMILES strings into its corresponding 42 dimensional MQN fingerprints. Now is time to transform those MQN fingerprints into PQ codes.

# Product Quantization

We fit the PQ encoder using our training data. My enamine data is already randomly mixed. If use the file directly as downlaoded from the enamine website make sure to properly select the chunk for training since those files come sorted by size

- `K`refers to the number of centroids to be used when running KMeans on each subvector. 

- `m`is the number of subvectors (splits) from our input data. 
 
- `iterations`is the maximum number of iterations each KMeans is going to do. 

With higher `K`and `iterations`, higher training times. 

In [ ]:
import numpy as np 

chunk = np.load('/mnt/samsung_2tb/tmp/enamine-chunk_00000.npy')
print(f"Shape: {(chunk.shape)}, MB: {chunk.nbytes/1e6}")


This is what a normal chunk of 10M MQN fingerprints take in space. In order to train the encoder that will transform MQN -> PQ codes, we can use 1 chunk since it will comfortably fit in memory. Feel free to load multiple chunks although 10M should be enough.

In [ ]:
from chelombus import PQEncoder
pq_encoder = PQEncoder(k=256, m=6, iterations=10)
pq_encoder.fit(chunk)


After training we can save/load the encoder

In [ ]:
import joblib
joblib.dump(pq_encoder, 'enamine_pqencoder_tutorial.joblib')

# To load it
pq_encoder = joblib.load('enamine_pqencoder_tutorial.joblib')


We can check some atributes: 

- `.codebook_cluster_centers` are the centroids coordinates gathered from each KMeans run on every subvector. Since we have 4 splits, 256 centroids and the subvectors are of size 1024/4 = 256, then the codebook is shape (4, 256, 256)

After the `pq_encoder` is trained, the encoder has an attribute to account for the training process. If we try to use transform without fitting we would get an Error. So know, we check that the ecoder was in fact trained. 

If we want to access all the `KMeans`attributes that one would normally get from sklearn, we can do so using `pq_trained` and use any attribute you would normally use. Like `.labels_` to check the index of the centroids for each training sample. 

In [ ]:
print("The shape of the codebook is: ", pq_encoder.codewords.shape)
print("Is the encoder trained? ", pq_encoder.encoder_is_trained)
print(f"The total number of lables {pq_encoder.pq_trained[0].labels_}  is: {len(pq_encoder.pq_trained[0].labels_)}")


After the training process we can create our PQ codes.
The PQCodes are going to be of shape `(Number of samples, m)`. 

In [ ]:
pq_codes = pq_encoder.transform(chunk)
print(f"Shape: {pq_codes.shape}, MB: {pq_codes.nbytes/1e6}")


# Inverse Transformation
We can also transform PQ codes to the original MQN vector and see how lossly the process is

In [ ]:
# Load a different chunk
mqn_chunk_2 = np.load('/mnt/samsung_2tb/tmp/enamine-chunk_00001.npy')

# Now that we trained the encoder we can convert MQN into PQCode and also test how lossy our compressio method is comparing the original MQN matrix vs the reconstructed one

ogMQN = mqn_chunk_2[:1_000_000]
print(ogMQN[:1].reshape(42,), ogMQN.shape)

pqcode = pq_encoder.transform(ogMQN)
print(pqcode[:1].reshape(6, ))

rMQN = pq_encoder.inverse_transform(pqcode)
print(rMQN[:1].reshape(42,), rMQN.shape)

print('Reconversion process:')

# MSE / RMSE
mse  = np.mean((ogMQN - rMQN)**2)
rmse = np.sqrt(mse)

def fidelity_frobenius(X, X_hat):
    num = np.linalg.norm(X - X_hat, 'fro')
    den = np.linalg.norm(X, 'fro')
    return (1 - num/den) * 100

score = fidelity_frobenius(ogMQN, rMQN)
print(f"Reconstruction fidelity: {score:.2f}%")


# Cluster

Now to the clustering part

In [ ]:
from chelombus import PQKMeans
import time

kmeans = PQKMeans(encoder=pq_encoder, k=10_000, iteration=20)
s = time.time()
kmeans.fit(pq_codes)
e = time.time()

print(f'Training with {len(pq_codes)} molecules took: {format_time(e-s)}')


We can of course save/load the Kmeans as well

In [ ]:
# joblib.dump(kmeans, 'kmeans_trained')
kmeans = joblib.load('kmeans_trained')


Finally, we can now get the cluster for every PQ code we pass

In [ ]:
kmeans.predict(pq_codes[:100])


In [ ]:

# Cluster the file with the SMILES strings
import os
import pandas as pd 
from chelombus import DataStreamer, FingerprintCalculator
from tqdm import tqdm

stream = DataStreamer()  
fp_calc = FingerprintCalculator()

output_path = '/mnt/samsung_2tb/tmp/'
for i, chunk in enumerate(stream.parse_input('/mnt/10tb_hdd/cleaned_enamine_10b/output_file_0.cxsmiles', chunksize=1_000_000, smiles_col=1)):
    chunk_pq_code = pq_encoder.transform(fp_calc.FingerprintFromSmiles(chunk, 'mqn'))
    labels = kmeans.predict(chunk_pq_code)
    df = pd.DataFrame({
        'smiles': chunk, 
        'cluster_id': labels
    })
    df.to_parquet(os.path.join(output_path, f'clustered_chunk_{i}.parquet'), index=False)
    


In [ ]:
# Cluster the file with the SMILES strings
from chelombus import PQEncoder, DataStreamer, FingerprintCalculator
from chelombus.clustering.PyQKmeans import PQKMeans
import pyarrow as pa, pyarrow.parquet as pq
from chelombus.utils import format_time
encoder = PQEncoder.load('../models/paper_encoder.joblib')
clusterer = PQKMeans.load('../models/paper_clusterer.joblib')
stream = DataStreamer()
fp_calc = FingerprintCalculator()
import timeit

for i, chunk in enumerate(stream.parse_input(
    '/mnt/10tb_hdd/cleaned_enamine_10b/output_file_0.cxsmiles',
    chunksize=1_000_000, smiles_col=1, verbose=0,
)):
    fps = fp_calc.FingerprintFromSmiles(chunk, 'mqn')
    t = timeit.timeit()
    pq_codes = encoder.transform(fps, verbose=0,  device='gpu')     
    print("GPU transform: ", timeit.timeit() - t)
    t = timeit.timeit()
    pq_codes = encoder.transform(fps, verbose=0,  device='cpu')     
    print("CPU transform: ", format_time(timeit.timeit() - t))
    t = timeit.timeit()
    labels = clusterer.predict(pq_codes, device='cpu')             
    print("CPU predict: ", format_time(timeit.timeit() - t))
    t = timeit.timeit()
    labels = clusterer.predict(pq_codes, device='gpu')             
    print("GPU predict: ", format_time(timeit.timeit() - t))
    # table = pa.table({"smiles": chunk, "cluster_id": labels})
    # pq.write_table(table, f'/mnt/samsung_2tb/tmp/chunk_{i:05d}.parquet')


# Visualization with TMAP

After clustering, we can create interactive TMAPs to explore the chemical space. The simplest approach: sample representatives from each cluster, compute MQN fingerprints, and let `TMAP` handle the layout.

In [ ]:
from chelombus import sample_from_cluster
from chelombus.utils.visualization import representative_tmap
# Sample representatives from several clusters
results_dir = '/mnt/2tb/tmp/'  # Path to your clustered parquet files
rep_smiles = []
rep_cids = []

for cid in range(100):  # First 100 clusters
    df = sample_from_cluster(results_dir, cluster_id=cid, n=1)
    if len(df) > 0:
        rep_smiles.extend(df['smiles'].tolist())
        rep_cids.extend([cid] * len(df))

print(f"Collected {len(rep_smiles)} representatives from {len(set(rep_cids))} clusters")


In [ ]:
# Create the representative TMAP, MQN + euclidean is the natural choice for representatives
representative_tmap(
    smiles=rep_smiles,
    cluster_ids=rep_cids,
    fingerprint='mqn',
    properties=['mw', 'logp', 'qed', 'n_rings', 'hba', 'hbd'],
    tmap_name='enamine_representative',
    k_neighbors=20,
)

print("Representative TMAP generated: enamine_representative.html")


You can also build a TMAP directly using tmap2 if you need more control:

```python
from tmap import TMAP
from tmap.utils import fingerprints_from_smiles, molecular_properties

fps = fingerprints_from_smiles(rep_smiles, fp_type='mqn')
model = TMAP(metric='euclidean', n_neighbors=20).fit(fps)

viz = model.to_tmapviz(include_edges=True)
props = molecular_properties(rep_smiles, properties=['mw', 'logp', 'qed'])
for name, values in props.items():
    viz.add_color_layout(name, values, categorical=False, color='rainbow')
viz.add_smiles(rep_smiles)
viz.write_html('custom_tmap.html')
```